# IPSR Unit 4: Knowledge and Reasoning

Subject: Introduction to Problem Solving and Reasoning (B.Tech Semester I).

Sixteen topics: from how knowledge is stored, to propositional and first-order logic, to planning. Everything is plain Python so you can see each step.

**How to use this notebook:** read a section, run every code cell with Shift+Enter, then try the exercises. Each exercise cell has only comments; write your answer there. Solutions are hidden: click *Show solution* only after you have tried.

---
## 4.1 Objectives, knowledge representation and reasoning, types of reasoning methods

*HTML lab: Unit 4 lab, lesson 1*

**Knowledge representation (KR)** is how facts and rules are stored so a program can use them. **Reasoning** derives new facts from stored ones.

| Method | Direction | Example |
|---|---|---|
| Deductive | general to specific, certain | All birds have wings; a sparrow is a bird; so a sparrow has wings |
| Inductive | specific to general, probable | Every swan I saw was white; so swans are white |
| Abductive | best explanation | The grass is wet; it probably rained |
| Common-sense | everyday defaults | Ice is cold |
| Monotonic / non-monotonic | new facts never / may retract old conclusions | Tweety is a bird so it flies; Tweety is a penguin so it does not |

In [1]:
kb = {"bird(sparrow)", "bird(penguin)", "penguin(penguin)"}
def flies(x):
    # non-monotonic default: birds fly unless known to be penguins
    return f"bird({x})" in kb and f"penguin({x})" not in kb
print("sparrow flies:", flies("sparrow"))
print("penguin flies:", flies("penguin"))

sparrow flies: True
penguin flies: False


**Exercise 1.** Label each as deductive, inductive or abductive: (a) the sun rose every day so far, so it will rise tomorrow; (b) the road is wet, so it must have rained; (c) all squares have 4 sides, this is a square, so it has 4 sides.

In [2]:
# Exercise 1: write your code below
# Hint: think direction of reasoning


<details><summary>Show solution</summary>

```python
answers = {"a": "inductive", "b": "abductive", "c": "deductive"}
print(answers)
```

</details>

---
## 4.2 Knowledge vs belief

*HTML lab: Unit 4 lab, lesson 1*

**Knowledge** is justified, true belief. **Belief** is what an agent holds to be true; it may be wrong. A knowledge base really stores beliefs, and the agent must revise them when new evidence arrives.

In [3]:
world = {"door_open": False}
belief = {"door_open": True}            # the agent believes the door is open
print("belief matches world:", belief["door_open"] == world["door_open"])
# percept arrives: the agent sees the door is closed
belief["door_open"] = world["door_open"]
print("after revising belief:", belief)

belief matches world: False
after revising belief: {'door_open': False}


**Exercise 1.** Is 'I believe the exam is on Monday' knowledge? What must be true for it to count as knowledge? Answer in a comment.

In [4]:
# Exercise 1: write your code below
# Hint: justified and true


<details><summary>Show solution</summary>

```python
# It becomes knowledge only if the exam really is on Monday AND the belief has justification
# (for example, read on the official timetable).
```

</details>

---
## 4.3 Types of knowledge

*HTML lab: Unit 4 lab, lesson 1*

In [5]:
types = {
 "Declarative":  "facts about the world: Paris is the capital of France",
 "Procedural":   "how to do things: steps to solve a quadratic",
 "Meta":         "knowledge about knowledge: I am weak at probability",
 "Heuristic":    "rules of thumb: try the cheapest fix first",
 "Structural":   "relations between things: a car has an engine and wheels",
}
for k, v in types.items(): print(f"{k:12} {v}")

Declarative  facts about the world: Paris is the capital of France
Procedural   how to do things: steps to solve a quadratic
Meta         knowledge about knowledge: I am weak at probability
Heuristic    rules of thumb: try the cheapest fix first
Structural   relations between things: a car has an engine and wheels


**Exercise 1.** Classify: 'To boil rice, add water and heat for 15 minutes.'

In [6]:
# Exercise 1: write your code below
# Hint: facts or steps?


<details><summary>Show solution</summary>

```python
print("Procedural")
```

</details>

---
## 4.4 Logical agents, knowledge-based agents

*HTML lab: Unit 4 lab, lesson 2*

A **knowledge-based agent** keeps a knowledge base (KB). Loop: **TELL** the KB what it perceives, **ASK** the KB what to do, **TELL** the KB the action it took.

In [7]:
class KBAgent:
    def __init__(self):
        self.kb = set()
        self.t = 0
    def tell(self, sentence): self.kb.add(sentence)
    def ask(self):
        if "smell(glitter)" in self.kb: return "grab"
        if "breeze" in self.kb:         return "go_back"
        return "move_forward"
    def step(self, percept):
        self.tell(percept)                  # TELL: percept
        action = self.ask()                 # ASK: what now?
        self.tell(f"did({action}, t={self.t})")   # TELL: action
        self.t += 1
        return action
a = KBAgent()
for p in ["nothing", "breeze", "smell(glitter)"]:
    a.kb = {p}                              # new percept only, to keep the demo simple
    print(p, "->", a.step(p))

nothing -> move_forward
breeze -> go_back
smell(glitter) -> grab


**Exercise 1.** Add a rule: if the percept is 'stench', the agent should return 'shoot'.

In [8]:
# Exercise 1: write your code below
# Hint: edit ask()


<details><summary>Show solution</summary>

```python
class KBAgent2(KBAgent):
    def ask(self):
        if "stench" in self.kb: return "shoot"
        return super().ask()
b = KBAgent2(); b.kb = {"stench"}
print(b.ask())
```

</details>

---
## 4.5 Propositional logic, truth table

*HTML lab: Unit 4 lab, lesson 3*

Propositions are True/False statements. Connectives: NOT, AND, OR, IMPLIES (`P -> Q` is false only when P is true and Q false), IFF.

In [9]:
from itertools import product
def table(expr, names):
    print(" ".join(f"{x:5}" for x in names), "|", expr)
    for vals in product([True, False], repeat=len(names)):
        env = dict(zip(names, vals))
        r = eval(expr, {}, {**env, "implies": lambda a, b: (not a) or b, "iff": lambda a, b: a == b})
        print(" ".join(f"{str(v):5}" for v in vals), "|", r)
table("implies(P, Q)", ["P", "Q"])
print()
table("(not (P and Q)) == ((not P) or (not Q))", ["P", "Q"])     # De Morgan

P     Q     | implies(P, Q)
True  True  | True
True  False | False
False True  | True
False False | True

P     Q     | (not (P and Q)) == ((not P) or (not Q))
True  True  | True
True  False | True
False True  | True
False False | True


**Exercise 1.** Show with a truth table that `implies(P, Q)` is equivalent to `(not P) or Q`.

In [10]:
# Exercise 1: write your code below
# Hint: the two columns match


<details><summary>Show solution</summary>

```python
ok = all(((not p) or q) == ((not p) or q) for p, q in product([True, False], repeat=2))
table("implies(P, Q) == ((not P) or Q)", ["P", "Q"])
```

</details>

---
## 4.6 Pros and cons of propositional logic

*HTML lab: Unit 4 lab, lesson 3*

**Pros:** simple, precise, decidable (a truth table always settles it), easy to code.
**Cons:** cannot say "all students" or "some student" so you need one proposition per object; a rule such as 'every square next to a pit is breezy' must be written once per square; cannot talk about relations. The size grows quickly.

In [11]:
squares = 16
print("propositional rules needed for 'pit next to square means breeze':", squares)
print("first-order logic needs: 1 rule  (for all x, y: Pit(x) and Adjacent(x, y) -> Breeze(y))")
print("truth table rows for 20 propositions:", 2**20)

propositional rules needed for 'pit next to square means breeze': 16
first-order logic needs: 1 rule  (for all x, y: Pit(x) and Adjacent(x, y) -> Breeze(y))
truth table rows for 20 propositions: 1048576


**Exercise 1.** How many truth-table rows for 10 propositions? For 30?

In [12]:
# Exercise 1: write your code below
# Hint: 2 ** n


<details><summary>Show solution</summary>

```python
print(2**10, 2**30)
```

</details>

---
## 4.7 Forward chaining

*HTML lab: Unit 4 lab, lesson 4*

Start from known facts and fire any rule whose conditions are all known, adding its conclusion, until nothing new appears (data-driven).

In [13]:
rules = [({"A", "B"}, "C"), ({"C"}, "D"), ({"D", "E"}, "F"), ({"X"}, "Y")]
facts = {"A", "B", "E"}
def forward(rules, facts, show=True):
    facts = set(facts); changed = True
    while changed:
        changed = False
        for cond, concl in rules:
            if cond <= facts and concl not in facts:
                facts.add(concl); changed = True
                if show: print(f"fire {sorted(cond)} -> {concl}   facts = {sorted(facts)}")
    return facts
result = forward(rules, facts)
print("F derived?", "F" in result)

fire ['A', 'B'] -> C   facts = ['A', 'B', 'C', 'E']
fire ['C'] -> D   facts = ['A', 'B', 'C', 'D', 'E']
fire ['D', 'E'] -> F   facts = ['A', 'B', 'C', 'D', 'E', 'F']
F derived? True


**Exercise 1.** Remove E from the facts. Can F still be derived?

In [14]:
# Exercise 1: write your code below
# Hint: run again


<details><summary>Show solution</summary>

```python
print("F" in forward(rules, {"A", "B"}, show=False))
```

</details>

---
## 4.8 Backward chaining

*HTML lab: Unit 4 lab, lesson 4*

Start from the goal and ask which rule concludes it, then try to prove that rule's conditions (goal-driven).

In [15]:
def backward(goal, rules, facts, depth=0):
    pad = "  " * depth
    if goal in facts:
        print(f"{pad}{goal}: known fact"); return True
    for cond, concl in rules:
        if concl == goal:
            print(f"{pad}{goal}: try rule {sorted(cond)} -> {goal}")
            if all(backward(c, rules, facts, depth + 1) for c in cond):
                return True
    print(f"{pad}{goal}: cannot prove"); return False
print("Proved F:", backward("F", rules, {"A", "B", "E"}))
print()
print("Proved Y:", backward("Y", rules, {"A", "B", "E"}))

F: try rule ['D', 'E'] -> F
  E: known fact
  D: try rule ['C'] -> D
    C: try rule ['A', 'B'] -> C
      A: known fact
      B: known fact
Proved F: True

Y: try rule ['X'] -> Y
  X: cannot prove
Y: cannot prove
Proved Y: False


**Exercise 1.** Prove D using backward chaining from facts {A, B}.

In [16]:
# Exercise 1: write your code below
# Hint: call backward


<details><summary>Show solution</summary>

```python
print(backward("D", rules, {"A", "B"}))
```

</details>

---
## 4.9 Forward vs backward chaining

*HTML lab: Unit 4 lab, lesson 4*

In [17]:
import time
big_rules = [({f"p{i}"}, f"p{i+1}") for i in range(300)] + [({"a"}, "b"), ({"b"}, "c")]
import io, contextlib
t0 = time.perf_counter(); r1 = forward(big_rules, {"p0", "a"}, show=False); t1 = time.perf_counter()
with contextlib.redirect_stdout(io.StringIO()):
    r2 = backward("c", big_rules, {"a"})
t2 = time.perf_counter()
print("forward chaining derived", len(r1), "facts (works on everything it can)")
print("backward chaining proved c, touching only the rules it needed:", r2)

forward chaining derived 304 facts (works on everything it can)
backward chaining proved c, touching only the rules it needed: True


| | Forward | Backward |
|---|---|---|
| Starts from | known facts | the goal |
| Good when | many facts arrive and you want all consequences (monitoring, alarms) | you have a specific question (diagnosis, 'can I prove X?') |
| Risk | derives many irrelevant facts | may loop on cyclic rules |

**Exercise 1.** Which suits a medical 'Does the patient have flu?' question better? Answer in a comment.

In [18]:
# Exercise 1: write your code below
# Hint: one goal in mind


<details><summary>Show solution</summary>

```python
# Backward chaining: it starts from the single hypothesis 'flu' and asks only for the symptoms that matter.
```

</details>

---
## 4.10 First-order logic, pros of FOL, logics in general

*HTML lab: Unit 4 lab, lesson 5*

FOL talks about **objects**, **relations** and **functions**, with quantifiers *for all* and *there exists*. It is more expressive than propositional logic: "All humans are mortal" is one sentence.

| Logic | World made of | Truth values |
|---|---|---|
| Propositional | facts | true / false |
| First-order | objects, relations | true / false |
| Temporal | objects, relations, time | true / false |
| Probabilistic | facts | degree of belief 0 to 1 |
| Fuzzy | facts with degree of truth | value 0 to 1 |

In [19]:
domain = ["socrates", "plato", "rex"]
Human  = {"socrates", "plato"}
Mortal = {"socrates", "plato", "rex"}
# For all x: Human(x) -> Mortal(x)
print(all((x not in Human) or (x in Mortal) for x in domain))

True


**Exercise 1.** Remove 'plato' from Mortal and check the rule again.

In [20]:
# Exercise 1: write your code below
# Hint: set discard


<details><summary>Show solution</summary>

```python
Mortal.discard("plato")
print(all((x not in Human) or (x in Mortal) for x in domain))
```

</details>

---
## 4.11 Syntax and components of FOL

*HTML lab: Unit 4 lab, lesson 6*

Components: **constants** (Socrates), **variables** (x), **predicates** (Human(x)), **functions** (FatherOf(x)), **connectives**, **quantifiers** (for all, exists), **equality**.

Common patterns: *For all x: P(x) -> Q(x)* (every P is Q) and *Exists x: P(x) and Q(x)* (some P is Q). Do not mix them: `Exists x: P(x) -> Q(x)` is almost always a mistake.

In [21]:
sentences = [
 ("Every student likes pizza",  "forall x: Student(x) -> Likes(x, Pizza)"),
 ("Some student likes pizza",   "exists x: Student(x) and Likes(x, Pizza)"),
 ("No student likes exams",     "forall x: Student(x) -> not Likes(x, Exams)"),
 ("Ravi's father is a teacher", "Teacher(FatherOf(Ravi))"),
]
for english, fol in sentences:
    print(f"{english:30} {fol}")

Every student likes pizza      forall x: Student(x) -> Likes(x, Pizza)
Some student likes pizza       exists x: Student(x) and Likes(x, Pizza)
No student likes exams         forall x: Student(x) -> not Likes(x, Exams)
Ravi's father is a teacher     Teacher(FatherOf(Ravi))


**Exercise 1.** Write in FOL: 'Every dog is an animal'.

In [22]:
# Exercise 1: write your code below
# Hint: forall + implies


<details><summary>Show solution</summary>

```python
print("forall x: Dog(x) -> Animal(x)")
```

</details>

---
## 4.12 Truth in FOL

*HTML lab: Unit 4 lab, lesson 6*

A sentence is true or false **in a model**: a set of objects plus what each predicate means. Below we write a tiny model and check quantified sentences by looking at every object.

In [23]:
model = {
  "domain": ["asha", "ravi", "meera"],
  "Student": {"asha", "ravi", "meera"},
  "Likes_Pizza": {"asha", "meera"},
}
D = model["domain"]
forall_ = all((x not in model["Student"]) or (x in model["Likes_Pizza"]) for x in D)
exists_ = any((x in model["Student"]) and (x in model["Likes_Pizza"]) for x in D)
print("every student likes pizza:", forall_)
print("some student likes pizza :", exists_)

every student likes pizza: False
some student likes pizza : True


**Exercise 1.** Check 'no student likes pizza' in the same model.

In [24]:
# Exercise 1: write your code below
# Hint: not exists


<details><summary>Show solution</summary>

```python
print(not any(x in model["Likes_Pizza"] for x in D))
```

</details>

---
## 4.13 First-order logic example

*HTML lab: Unit 4 lab, lesson 6*

Classic example. Knowledge: all humans are mortal; Socrates is human. Question: is Socrates mortal? Here we prove it by instantiating the rule for the constant Socrates (universal instantiation) and then applying modus ponens.

In [25]:
kb_facts = {("Human", "Socrates")}
kb_rules = [("Human", "Mortal")]            # forall x: Human(x) -> Mortal(x)
def infer(facts, rules):
    facts = set(facts)
    changed = True
    while changed:
        changed = False
        for p, q in rules:
            for (pred, obj) in list(facts):
                if pred == p and (q, obj) not in facts:
                    facts.add((q, obj)); changed = True
                    print(f"{p}({obj}) and [forall x: {p}(x) -> {q}(x)]  =>  {q}({obj})")
    return facts
print(("Mortal", "Socrates") in infer(kb_facts, kb_rules))

Human(Socrates) and [forall x: Human(x) -> Mortal(x)]  =>  Mortal(Socrates)
True


**Exercise 1.** Add the fact Human(Plato) and the rule Mortal -> WillDie. Infer everything.

In [26]:
# Exercise 1: write your code below
# Hint: append to sets/lists


<details><summary>Show solution</summary>

```python
kb_facts.add(("Human", "Plato")); kb_rules.append(("Mortal", "WillDie"))
print(sorted(infer(kb_facts, kb_rules)))
```

</details>

---
## 4.14 Classical planning, introduction

*HTML lab: Unit 4 lab, lesson 7*

Classical planning finds a sequence of actions from a start state to a goal in an environment that is fully observable, deterministic, finite and static. Each **action** has *preconditions* and *effects*. A **plan** is the action sequence.

In [27]:
from collections import deque
# Tiny world: a robot with a key moves between rooms. State = (robot_room, key_room, holding)
def actions(state):
    r, k, h = state
    out = []
    for nr in ("Hall", "Lab"):
        if nr != r: out.append((f"Go({nr})", (nr, nr if h else k, h)))
    if not h and r == k: out.append(("Pick(key)", (r, k, True)))
    if h: out.append(("Drop(key)", (r, r, False)))
    return out
def plan(start, goal_test):
    q = deque([(start, [])]); seen = {start}
    while q:
        s, p = q.popleft()
        if goal_test(s): return p
        for name, ns in actions(s):
            if ns not in seen: seen.add(ns); q.append((ns, p + [name]))
print(plan(("Hall", "Lab", False), lambda s: s[1] == "Hall"))

['Go(Lab)', 'Pick(key)', 'Go(Hall)']


**Exercise 1.** Start with the robot and the key both in the Hall. Find a plan that ends with the key in the Lab and the robot back in the Hall.

In [28]:
# Exercise 1: write your code below
# Hint: goal_test checks two fields


<details><summary>Show solution</summary>

```python
print(plan(("Hall", "Hall", False), lambda s: s[0] == "Hall" and s[1] == "Lab"))
```

</details>

---
## 4.15 Search vs planning

*HTML lab: Unit 4 lab, lesson 7*

| | Ordinary search | Planning |
|---|---|---|
| States | black boxes, tested by a goal function | described by logical facts |
| Actions | coded as a successor function | described by preconditions and effects |
| Heuristics | written by hand for each problem | derived automatically from the description |
| Goal | tested only | described, so the planner can reason about it |
| Scale | struggles when the state space is huge | can use structure to cut the search |

In [29]:
# Count how big a state space gets: 8 blocks stacked in any arrangement
from math import factorial
for blocks in (3, 5, 8, 10):
    print(blocks, "blocks -> at least", f"{factorial(blocks):,}", "orderings of a single tower")

3 blocks -> at least 6 orderings of a single tower
5 blocks -> at least 120 orderings of a single tower
8 blocks -> at least 40,320 orderings of a single tower
10 blocks -> at least 3,628,800 orderings of a single tower


**Exercise 1.** Why is describing actions by preconditions and effects helpful for a planner? One sentence in a comment.

In [30]:
# Exercise 1: write your code below
# Hint: think: generic code


<details><summary>Show solution</summary>

```python
# The planner can work out for itself what is applicable and what changes, so one generic solver handles many domains.
```

</details>

---
## 4.16 The language of planning problems (PDDL)

*HTML lab: Unit 4 lab, lesson 8*

PDDL describes a **domain** (predicates and actions) and a **problem** (objects, initial state, goal). Below is a blocks-world domain written as Python data in the same shape as PDDL, plus a small solver that reads it.

In [31]:
import itertools
# PDDL-like domain: each action has preconditions, add effects, delete effects
def blocks_actions(blocks):
    acts = {}
    for b in blocks:
        acts[f"pickup({b})"]  = ({f"clear({b})", f"ontable({b})", "handempty"}, {f"holding({b})"}, {f"clear({b})", f"ontable({b})", "handempty"})
        acts[f"putdown({b})"] = ({f"holding({b})"}, {f"clear({b})", f"ontable({b})", "handempty"}, {f"holding({b})"})
        for c in blocks:
            if b != c:
                acts[f"stack({b},{c})"]   = ({f"holding({b})", f"clear({c})"}, {f"on({b},{c})", f"clear({b})", "handempty"}, {f"holding({b})", f"clear({c})"})
                acts[f"unstack({b},{c})"] = ({f"on({b},{c})", f"clear({b})", "handempty"}, {f"holding({b})", f"clear({c})"}, {f"on({b},{c})", f"clear({b})", "handempty"})
    return acts
def solve(blocks, init, goal):
    acts = blocks_actions(blocks)
    start = frozenset(init); q = deque([(start, [])]); seen = {start}
    while q:
        s, p = q.popleft()
        if goal <= s: return p
        for name, (pre, add, dele) in acts.items():
            if pre <= s:
                ns = frozenset((s - dele) | add)
                if ns not in seen: seen.add(ns); q.append((ns, p + [name]))
init = {"ontable(A)", "on(B,A)", "clear(B)", "ontable(C)", "clear(C)", "handempty"}
goal = {"on(A,B)"}
for i, step in enumerate(solve("ABC", init, goal), 1): print(i, step)

1 unstack(B,A)
2 putdown(B)
3 pickup(A)
4 stack(A,B)


**Exercise 1.** Change the goal to {'on(B,C)'} and solve again.

In [32]:
# Exercise 1: write your code below
# Hint: only the goal changes


<details><summary>Show solution</summary>

```python
for i, step in enumerate(solve("ABC", init, {"on(B,C)"}), 1): print(i, step)
```

</details>

**Where to read more:** AIMA chapters 7 to 10 (logic and planning); planning.domains for real PDDL examples